# ForecastAI — Fine-tune QLoRA (Model 2)

**Trước khi chạy notebook này:**
1. Bật GPU: Settings (bên phải) → Accelerator → **GPU T4 x2**.
2. Upload dataset lên Kaggle Datasets (thư mục `data/llm_dataset` từ máy bạn: `train.jsonl`, `validation.jsonl`, `test.jsonl`, `dataset_info.json`, `review_sample.md`), rồi bấm **Add Input** ở panel bên phải, chọn dataset đó.
3. (Chỉ cần nếu chạy Llama) Vào Settings → Secrets, thêm secret tên `HF_TOKEN` = Hugging Face access token của bạn (đã accept license Llama-3.1-8B-Instruct trên huggingface.co trước).

Repo: https://github.com/lephiangit/Forecastmoney

In [ ]:
# Kiểm tra GPU đã bật chưa — nếu lệnh này lỗi, quay lại Settings bật Accelerator trước khi chạy tiếp.
!nvidia-smi

In [ ]:
# Clone code (KHÔNG chứa dataset — data/ nằm trong .gitignore, dataset lấy từ Kaggle Input đã add ở bước 2).
!git clone https://github.com/lephiangit/Forecastmoney.git ForecastAI
%cd ForecastAI

In [ ]:
!pip install -q transformers peft bitsandbytes accelerate trl datasets

## Bước 0 — Tải trước model gốc (tách riêng khỏi bước train)

Model KHÔNG có sẵn trên máy bạn hay trên GitHub — vì nó ~15GB, vượt xa giới hạn file của GitHub, và máy không GPU thì tải về cũng vô ích. Bước này tải và cache model ngay trên Kaggle, tách riêng khỏi bước train để nếu mạng đứt giữa chừng thì không mất giờ GPU đã dùng.

In [ ]:
!pip install -q huggingface_hub
!python -m training.download_base_models --model qwen

In [ ]:
# Xác nhận dataset đã được Kaggle mount đúng chỗ — đổi DATASET_NAME nếu bạn đặt tên khác lúc upload.
import os

DATASET_NAME = "forecastai-llm-dataset"
DATASET_PATH = f"/kaggle/input/{DATASET_NAME}"

print("Nội dung /kaggle/input:", os.listdir("/kaggle/input"))
assert os.path.exists(DATASET_PATH), f"Không thấy {DATASET_PATH} — kiểm tra lại tên dataset đã Add Input."
print("OK, tìm thấy:", os.listdir(DATASET_PATH))

## Bước 1 — Fine-tune Qwen2.5-7B (khuyến nghị chạy trước — không cần xin quyền HF)

In [ ]:
!python -m training.finetune_qlora \
  --model qwen \
  --dataset /kaggle/input/forecastai-llm-dataset \
  --output /kaggle/working/llm_adapter_qwen \
  --epochs 3

Ước lượng ~2-4 giờ trên T4 với 3000 mẫu, 3 epoch. Ghi lại `trainable_params`/`total_params` và `final_train_loss` in ra cuối — cần cho báo cáo.

In [ ]:
# Sinh output so sánh TRƯỚC / SAU fine-tune trên tập test — dùng cho phần "loss 0.8 nghĩa là gì" khi bảo vệ đồ án.
!python -m training.generate_comparison_samples \
  --model qwen \
  --adapter /kaggle/working/llm_adapter_qwen \
  --dataset /kaggle/input/forecastai-llm-dataset \
  --n 8

In [ ]:
# Đọc nhanh vài mẫu so sánh ngay trong notebook trước khi tải file về.
with open("/kaggle/working/llm_adapter_qwen/before_after_comparison.md", encoding="utf-8") as f:
    print(f.read()[:4000])

## Bước 2 (tuỳ chọn) — Fine-tune Llama-3.1-8B để có số liệu so sánh hai model

Cần: đã accept license tại https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct, và đã thêm secret `HF_TOKEN` (Settings → Secrets).

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)

In [ ]:
!python -m training.finetune_qlora \
  --model llama \
  --dataset /kaggle/input/forecastai-llm-dataset \
  --output /kaggle/working/llm_adapter_llama \
  --epochs 3

In [ ]:
!python -m training.generate_comparison_samples \
  --model llama \
  --adapter /kaggle/working/llm_adapter_llama \
  --dataset /kaggle/input/forecastai-llm-dataset \
  --n 8

## Bước 3 — Nén adapter để tải về (hoặc đẩy thẳng lên Hugging Face Hub ở bước dưới)

In [ ]:
# File adapter chỉ vài chục MB (LoRA, không phải full model) — nén lại để tải về máy kèm báo cáo.
!zip -r /kaggle/working/llm_adapter_qwen.zip /kaggle/working/llm_adapter_qwen
print("Xong — tải file .zip này ở panel Output bên phải.")

## Bước 4 (tuỳ chọn) — Đẩy adapter lên Hugging Face Hub

Cần `HF_TOKEN` như bước Llama ở trên. Đổi `ten-cua-ban/forecastai-research` thành namespace Hugging Face thật của bạn.

In [ ]:
!python -m training.finetune_qlora \
  --model qwen \
  --dataset /kaggle/input/forecastai-llm-dataset \
  --output /kaggle/working/llm_adapter_qwen \
  --epochs 3 \
  --push-to-hub ten-cua-ban/forecastai-research